# Studi Komparatif Model Generatif: AE, VAE, GAN, DCGAN, DDPM

Perbandingan enam model generatif pada Fashion-MNIST, plus eksperimen
pembuktian keterbatasan bobot dan latent feature MLP classifier untuk
image generation. Eksperimen dijalankan pada GPU (Colab T4).

---
# BAGIAN 0 — Setup, Config & Data

In [ ]:
import time, os, math, json, random, hashlib
from pathlib import Path
PROJECT_ROOT = globals().get("PROJECT_ROOT", Path.cwd())
import numpy as np
import scipy.linalg
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import transforms
from torch.utils.data import DataLoader, ConcatDataset, random_split
import matplotlib.pyplot as plt
import pandas as pd
from skimage.metrics import structural_similarity as ssim_fn

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)
os.makedirs("checkpoints", exist_ok=True)
os.makedirs("results/logs", exist_ok=True)
os.makedirs("results/figures", exist_ok=True)
print(f"device={device} | torch={torch.__version__} | colab={IN_COLAB}")
if device.type == "cuda":
    print(torch.cuda.get_device_name(0))

In [ ]:
# ===== CONFIG — Ready-to-Use (QUICK=False untuk eksperimen penuh) =====
QUICK = False

if QUICK:
    EP = dict(mlp=1, ae=1, sparse=1, vae=1, gan=1, dcgan=1, ddpm=1, fdec=1)
    FID_N = 64
else:
    EP = dict(mlp=30, ae=30, sparse=30, vae=30, gan=50, dcgan=40, ddpm=25, fdec=30)
    FID_N = 1000

CFG = dict(
    batch_size   = 256,
    latent       = 64,     # VanillaAE / VAE / z_MLP
    sparse_lat   = 256,    # SparseAE latent (sesuai template rubrik "256 (Sparse)")
    gan_z        = 100,    # prior noise GAN & DCGAN
    lam_sparse   = 1e-3,   # lambda penalti L1 SparseAE
    lr_ae        = 1e-3,
    lr_gan       = 2e-4,   # Adam betas (0.5, 0.999) utk GAN/DCGAN
    lr_ddpm      = 2e-4,
    T            = 1000,   # langkah difusi DDPM
    beta_start   = 1e-4, beta_end = 0.02,
    n_fig1       = 8,
)
RUN_CONFIG = dict(version=3, seed=42, quick=QUICK, epochs=EP, fid_n=FID_N, cfg=CFG)
config_id = hashlib.sha256(json.dumps(RUN_CONFIG, sort_keys=True).encode()).hexdigest()[:12]
RUN_DIR = PROJECT_ROOT / 'runs' / f"{'quick' if QUICK else 'full'}_{config_id}"
RUN_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(RUN_DIR)
for folder in ['checkpoints', 'results/logs', 'results/figures']:
    Path(folder).mkdir(parents=True, exist_ok=True)
Path('results/config.json').write_text(json.dumps(RUN_CONFIG, indent=2))
print({"QUICK": QUICK, **EP, "FID_N": FID_N, "output": str(RUN_DIR)})

In [ ]:
# ===== Data: Fashion-MNIST 70k -> 80/10/10 (56k/7k/7k), seed 42 =====
tfm = transforms.ToTensor()  # [0,1]
ds_full = ConcatDataset([
    torchvision.datasets.FashionMNIST("./data", train=True,  download=True, transform=tfm),
    torchvision.datasets.FashionMNIST("./data", train=False, download=True, transform=tfm),
])
g = torch.Generator().manual_seed(42)
ds_train, ds_val, ds_test = random_split(ds_full, [56000, 7000, 7000], generator=g)
assert len(ds_train) == 56000 and len(ds_val) == 7000 and len(ds_test) == 7000

B = CFG["batch_size"]
NW = 0 if os.name == "nt" else 2   # num_workers>0 di Windows+notebook bisa hang
train_loader = DataLoader(ds_train, batch_size=B, shuffle=True,  num_workers=NW, pin_memory=True)
val_loader   = DataLoader(ds_val,   batch_size=B, shuffle=False, num_workers=NW, pin_memory=True)
test_loader  = DataLoader(ds_test,  batch_size=B, shuffle=False, num_workers=NW, pin_memory=True)

CLASSES = ["T-shirt","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Boot"]

x, y = next(iter(test_loader))
grid = torchvision.utils.make_grid(x[:16], nrow=4).permute(1, 2, 0)
plt.figure(figsize=(5, 5)); plt.imshow(grid, cmap="gray"); plt.axis("off")
plt.title("Fashion-MNIST test samples"); plt.show()
print("split:", len(ds_train), len(ds_val), len(ds_test))

---
# BAGIAN 1 — Arsitektur Model

Normalisasi: dataset `[0,1]`; GAN/DCGAN/DDPM memakai `x*2-1 → [-1,1]` secara internal
(tanh output / noise prediction), AE-family & VAE pakai Sigmoid output di `[0,1]`.

In [ ]:
# ===== Utils: checkpoint & CSV logger =====
def save_ckpt(model, name):
    torch.save(model.state_dict(), f"checkpoints/{name}.pt")

def load_ckpt(model, name):
    model.to(device)
    model.load_state_dict(torch.load(f"checkpoints/{name}.pt", map_location=device, weights_only=True))
    return model.eval()

def training_done(tag):
    return Path(f'checkpoints/{tag}.done').exists()

def mark_done(tag):
    Path(f'checkpoints/{tag}.done').touch()

class CSVLogger:
    def __init__(self, name):
        self.path = f"results/logs/{name}.csv"; self.rows = []
    def log(self, **kw):
        self.rows.append(kw)
        pd.DataFrame(self.rows).to_csv(self.path, index=False)

def show_grid(imgs, nrow=8, title="", ax=None, size=4):
    g = torchvision.utils.make_grid(imgs.clamp(0, 1).cpu(), nrow=nrow).permute(1, 2, 0).squeeze(-1)
    if ax is None:
        _, ax = plt.subplots(figsize=(size, size))
    ax.imshow(g, cmap="gray", vmin=0, vmax=1); ax.axis("off"); ax.set_title(title)
    return ax

In [ ]:
# ===== Vanilla & Sparse Autoencoder (MLP) =====
class VanillaAE(nn.Module):
    """Enc 784->256->latent | Dec latent->256->784(sigmoid)"""
    def __init__(self, latent=64):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, latent))
        self.decoder = nn.Sequential(
            nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def encode(self, x):
        return self.encoder(x.view(x.size(0), -1))
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)
    def forward(self, x):
        return self.decode(self.encode(x))

class SparseAE(VanillaAE):
    """Sama seperti VanillaAE; loss ditambah penalti L1 pada latent (di trainer)."""
    pass

In [ ]:
# ===== VAE (MLP) =====
class VAE(nn.Module):
    def __init__(self, latent=64):
        super().__init__()
        self.fc1  = nn.Linear(784, 256)
        self.mu   = nn.Linear(256, latent)
        self.lv   = nn.Linear(256, latent)
        self.decoder = nn.Sequential(
            nn.Linear(latent, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def encode(self, x):
        h = F.relu(self.fc1(x.view(x.size(0), -1)))
        return self.mu(h), self.lv(h)
    def reparam(self, mu, lv):
        return mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
    def decode(self, z):
        return self.decoder(z).view(-1, 1, 28, 28)
    def forward(self, x):
        mu, lv = self.encode(x)
        return self.decode(self.reparam(mu, lv)), mu, lv

def vae_loss(recon, x, mu, lv):
    B = x.size(0)
    rec = F.mse_loss(recon, x, reduction="sum") / B          # per-image summed-pixel MSE
    kl  = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp()) / B
    return rec + kl, rec, kl

In [ ]:
# ===== Vanilla GAN (MLP) =====
class GAN_G(nn.Module):
    def __init__(self, z=100):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(z, 256), nn.ReLU(),
            nn.Linear(256, 512), nn.ReLU(),
            nn.Linear(512, 784), nn.Tanh())
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

class GAN_D(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(784, 512), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(512, 256), nn.LeakyReLU(0.2), nn.Dropout(0.3),
            nn.Linear(256, 1))
    def forward(self, x):
        return self.net(x.view(x.size(0), -1)).view(-1)

def weights_init(m):
    if isinstance(m, (nn.Linear, nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0, 0.02)
        if m.bias is not None: nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02); nn.init.zeros_(m.bias)

In [ ]:
# ===== DCGAN (conv, 28x28) =====
class DCGAN_G(nn.Module):
    def __init__(self, z=100):
        super().__init__()
        self.fc = nn.Sequential(nn.Linear(z, 512 * 7 * 7), nn.BatchNorm1d(512 * 7 * 7), nn.ReLU())
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(512, 256, 4, 2, 1), nn.BatchNorm2d(256), nn.ReLU(),   # 7->14
            nn.ConvTranspose2d(256, 1, 4, 2, 1), nn.Tanh())                          # 14->28
    def forward(self, z):
        h = self.fc(z).view(-1, 512, 7, 7)
        return self.deconv(h)

class DCGAN_D(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1), nn.LeakyReLU(0.2),                            # 28->14
            nn.Conv2d(64, 128, 4, 2, 1), nn.BatchNorm2d(128), nn.LeakyReLU(0.2))     # 14->7
        self.head = nn.Linear(128 * 7 * 7, 1)
    def forward(self, x):
        return self.head(self.conv(x).flatten(1)).view(-1)

In [ ]:
# ===== DDPM: sinusoidal time-emb + ResBlock + UNet kecil =====
class TimeEmb(nn.Module):
    def __init__(self, dim=256):
        super().__init__(); self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim // 4, dim), nn.SiLU(), nn.Linear(dim, dim))
    def forward(self, t):
        half = self.dim // 8
        f = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        ang = t[:, None].float() * f[None]
        return self.mlp(torch.cat([ang.sin(), ang.cos()], dim=1))

class ResBlock(nn.Module):
    def __init__(self, cin, cout, tdim=256):
        super().__init__()
        self.c1 = nn.Sequential(nn.GroupNorm(8, cin), nn.SiLU(), nn.Conv2d(cin, cout, 3, 1, 1))
        self.tp = nn.Linear(tdim, cout)
        self.c2 = nn.Sequential(nn.GroupNorm(8, cout), nn.SiLU(), nn.Conv2d(cout, cout, 3, 1, 1))
        self.skip = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()
    def forward(self, x, t):
        h = self.c1(x) + self.tp(t)[:, :, None, None]
        return self.c2(h) + self.skip(x)

class UNet(nn.Module):
    def __init__(self, tdim=256):
        super().__init__()
        self.tm = TimeEmb(tdim)
        self.inc = nn.Conv2d(1, 64, 3, 1, 1)
        self.d1 = ResBlock(64, 64, tdim);    self.dn1 = nn.Conv2d(64, 64, 3, 2, 1)     # 28->14
        self.d2 = ResBlock(64, 128, tdim);   self.dn2 = nn.Conv2d(128, 128, 3, 2, 1)   # 14->7
        self.mid = ResBlock(128, 256, tdim)
        self.up2 = nn.ConvTranspose2d(256, 128, 4, 2, 1)                               # 7->14
        self.u2 = ResBlock(256, 128, tdim)
        self.up1 = nn.ConvTranspose2d(128, 64, 4, 2, 1)                                # 14->28
        self.u1 = ResBlock(128, 64, tdim)
        self.out = nn.Conv2d(64, 1, 3, 1, 1)
    def forward(self, x, t):
        t = self.tm(t)
        h1 = self.d1(self.inc(x), t)                    # 64 @28
        h2 = self.d2(self.dn1(h1), t)                   # 128@14
        m  = self.mid(self.dn2(h2), t)                  # 256@7
        u  = self.u2(torch.cat([self.up2(m), h2], 1), t)  # (128+128)->128 @14
        u  = self.u1(torch.cat([self.up1(u), h1], 1), t)  # (64+64)->64  @28
        return self.out(u)

class GaussianDiffusion(nn.Module):
    """DDPM standar: linear beta schedule, prediksi noise eps, ancestral sampler."""
    def __init__(self, model, T=1000, b0=1e-4, b1=0.02):
        super().__init__()
        self.model, self.T = model, T
        betas = torch.linspace(b0, b1, T)
        ab = torch.cumprod(1 - betas, 0)
        self.register_buffer("betas", betas)
        self.register_buffer("ab", ab)
        self.register_buffer("ab_prev", F.pad(ab[:-1], (1, 0), value=1.0))

    def q_sample(self, x0, t, noise):
        a = self.ab[t][:, None, None, None]
        return a.sqrt() * x0 + (1 - a).sqrt() * noise

    def loss(self, x0):
        t = torch.randint(0, self.T, (x0.size(0),), device=x0.device)
        noise = torch.randn_like(x0)
        return F.mse_loss(self.model(self.q_sample(x0, t, noise), t), noise)

    @torch.no_grad()
    def p_sample(self, x, t):
        tb = torch.full((x.size(0),), t, device=x.device, dtype=torch.long)
        eps = self.model(x, tb)
        b, a, ab_t, ab_p = self.betas[t], 1 - self.betas[t], self.ab[t], self.ab_prev[t]
        x0_pred = ((x - (1 - ab_t).sqrt() * eps) / ab_t.sqrt()).clamp(-1.0, 1.0)
        mean = (ab_p.sqrt() * b / (1 - ab_t)) * x0_pred + (a.sqrt() * (1 - ab_p) / (1 - ab_t)) * x
        var = b * (1 - ab_p) / (1 - ab_t)
        noise = torch.randn_like(x) if t > 0 else 0
        return mean + var.sqrt() * noise

    @torch.no_grad()
    def sample(self, n, traj=False):
        x = torch.randn(n, 1, 28, 28, device=self.ab.device)
        self.eval()
        frames = [((x.clamp(-1, 1) + 1) / 2).cpu()] if traj else []
        marks = set(np.linspace(0, self.T - 1, 10).astype(int)) if traj else set()
        for t in reversed(range(self.T)):
            x = self.p_sample(x, t)
            if t in marks: frames.append(((x.clamp(-1, 1) + 1) / 2).cpu())
        out = (x.clamp(-1, 1) + 1) / 2
        return (out, frames) if traj else out   # frames sudah urut noisy -> bersih

In [ ]:
# ===== MLP Classifier (784->256->64->10) + FrozenDecoder =====
class MLPClassifier(nn.Module):
    """h1=ReLU(xW1+b1); z_MLP=ReLU(h1W2+b2); logits=zW3+b3 — z_MLP di R^64."""
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 256)
        self.fc2 = nn.Linear(256, 64)
        self.fc3 = nn.Linear(64, 10)
    def features(self, x):
        h1 = F.relu(self.fc1(x.view(x.size(0), -1)))
        return F.relu(self.fc2(h1))          # z_MLP (post-ReLU, non-negatif)
    def forward(self, x):
        return self.fc3(self.features(x))

class FrozenDecoder(nn.Module):
    """Decoder terpisah 64->256->784 utk rekonstruksi dari z_MLP beku (Tahap 4)."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 784), nn.Sigmoid())
    def forward(self, z):
        return self.net(z).view(-1, 1, 28, 28)

---
# BAGIAN 2 — Training

Loss dan waktu per epoch dicatat ke `results/logs/<model>.csv`, bobot ke
`checkpoints/<model>.pt`. Checkpoint dipilih dari validation set; test set hanya
untuk evaluasi akhir. Model dengan marker `.done` dilewati saat re-run.

In [ ]:
# ===== Trainers =====
def train_ae_like(model, tag, epochs, loss_fn):
    """loss_fn(model, x) -> (loss, extra_dict). Dipakai AE, SparseAE, VAE, FrozenDecoder."""
    if training_done(tag):
        print(f"[{tag}] checkpoint ada, skip"); return load_ckpt(model, tag)
    model.to(device).train()
    opt = torch.optim.Adam(model.parameters(), lr=CFG["lr_ae"])
    log = CSVLogger(tag)
    best_val = float('inf')
    for ep in range(1, epochs + 1):
        model.train()
        t0, tot = time.time(), 0.0
        for x, _ in train_loader:
            x = x.to(device)
            loss, _ = loss_fn(model, x)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        sec = time.time() - t0
        model.eval()
        val_total = 0.0
        with torch.no_grad():
            for x, _ in val_loader:
                loss, _ = loss_fn(model, x.to(device))
                val_total += loss.item() * x.size(0)
        val_loss = val_total / len(ds_val)
        if val_loss < best_val:
            best_val = val_loss
            save_ckpt(model, tag)
        log.log(epoch=ep, train_loss=tot / len(ds_train), val_loss=val_loss, sec=sec)
        print(f"[{tag}] ep {ep:3d}/{epochs} loss={tot/len(ds_train):.4f} ({sec:.1f}s)")
    mark_done(tag)
    return load_ckpt(model, tag)

def ae_loss(model, x):
    r = model(x); return F.mse_loss(r, x), {}

def sparse_loss(model, x):
    z = model.encode(x); r = model.decode(z)
    return F.mse_loss(r, x) + CFG["lam_sparse"] * z.abs().mean(), {}

def vae_loss_wrap(model, x):
    r, mu, lv = model(x)
    loss, rec, kl = vae_loss(r, x, mu, lv)
    return loss, {"rec": rec.item(), "kl": kl.item()}

def train_gan(G, D, tag, epochs):
    if training_done(tag):
        print(f"[{tag}] checkpoint ada, skip")
        return load_ckpt(G, f"{tag}_G"), load_ckpt(D, f"{tag}_D")
    G.to(device).train(); D.to(device).train()
    go = torch.optim.Adam(G.parameters(), lr=CFG["lr_gan"], betas=(0.5, 0.999))
    do = torch.optim.Adam(D.parameters(), lr=CFG["lr_gan"], betas=(0.5, 0.999))
    log = CSVLogger(tag)
    for ep in range(1, epochs + 1):
        t0, dg, gg = time.time(), 0.0, 0.0
        for x, _ in train_loader:
            x = (x.to(device) * 2 - 1)                       # [-1,1]
            n, z = x.size(0), CFG["gan_z"]
            # --- D step: max log D(real) + log(1 - D(G(z))), real label 0.9 (smoothing)
            fake = G(torch.randn(n, z, device=device)).detach()
            dl = F.binary_cross_entropy_with_logits(D(x), 0.9 * torch.ones(n, device=device)) + \
                 F.binary_cross_entropy_with_logits(D(fake), torch.zeros(n, device=device))
            do.zero_grad(); dl.backward(); do.step()
            # --- G step: max log D(G(z))
            gz = G(torch.randn(n, z, device=device))
            gl = F.binary_cross_entropy_with_logits(D(gz), torch.ones(n, device=device))
            go.zero_grad(); gl.backward(); go.step()
            dg += dl.item() * n; gg += gl.item() * n
        sec = time.time() - t0
        log.log(epoch=ep, d_loss=dg / len(ds_train), g_loss=gg / len(ds_train), sec=sec)
        print(f"[{tag}] ep {ep:3d}/{epochs} D={dg/len(ds_train):.3f} G={gg/len(ds_train):.3f} ({sec:.1f}s)")
    save_ckpt(G, f"{tag}_G"); save_ckpt(D, f"{tag}_D")
    mark_done(tag)
    return G.eval(), D.eval()

def train_ddpm(diff, epochs):
    if training_done('ddpm'):
        print("[ddpm] checkpoint ada, skip"); return load_ckpt(diff, "ddpm")
    diff.to(device).train()
    opt = torch.optim.Adam(diff.parameters(), lr=CFG["lr_ddpm"])
    log = CSVLogger("ddpm")
    for ep in range(1, epochs + 1):
        t0, tot = time.time(), 0.0
        for x, _ in train_loader:
            loss = diff.loss(x.to(device) * 2 - 1)
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        sec = time.time() - t0
        log.log(epoch=ep, train_loss=tot / len(ds_train), sec=sec)
        print(f"[ddpm] ep {ep:3d}/{epochs} loss={tot/len(ds_train):.4f} ({sec:.1f}s)")
    save_ckpt(diff, "ddpm")
    mark_done('ddpm')
    return diff.eval()

@torch.no_grad()
def test_acc(clf, loader):
    clf.eval()
    ok = sum((clf(x.to(device)).argmax(1) == y.to(device)).sum().item() for x, y in loader)
    return ok / len(loader.dataset)

def train_mlp(clf, epochs):
    if training_done('mlp'):
        print("[mlp] checkpoint ada, skip"); return load_ckpt(clf, "mlp")
    clf.to(device).train()
    opt = torch.optim.Adam(clf.parameters(), lr=1e-3)
    log = CSVLogger("mlp")
    best_acc = -1.0
    for ep in range(1, epochs + 1):
        t0, tot = time.time(), 0.0
        for x, y in train_loader:
            loss = F.cross_entropy(clf(x.to(device)), y.to(device))
            opt.zero_grad(); loss.backward(); opt.step()
            tot += loss.item() * x.size(0)
        acc = test_acc(clf, val_loader); clf.train()
        sec = time.time() - t0
        log.log(epoch=ep, train_loss=tot / len(ds_train), val_acc=acc, sec=sec)
        print(f"[mlp] ep {ep:3d}/{epochs} loss={tot/len(ds_train):.4f} val_acc={acc:.4f} ({sec:.1f}s)")
        if acc > best_acc:
            best_acc = acc
            save_ckpt(clf, 'mlp')
    mark_done('mlp')
    return load_ckpt(clf, 'mlp')

In [ ]:
# ===== Jalankan training semua model =====
ae     = train_ae_like(VanillaAE(CFG["latent"]),                    "ae",     EP["ae"],     ae_loss)
sparse = train_ae_like(SparseAE(CFG["sparse_lat"]),                 "sparse", EP["sparse"], sparse_loss)
vae    = train_ae_like(VAE(CFG["latent"]),                          "vae",    EP["vae"],    vae_loss_wrap)
ganG, ganD     = train_gan(GAN_G(CFG["gan_z"]).apply(weights_init),
                           GAN_D().apply(weights_init),             "gan",    EP["gan"])
dcgG, dcgD     = train_gan(DCGAN_G(CFG["gan_z"]).apply(weights_init),
                           DCGAN_D().apply(weights_init),           "dcgan",  EP["dcgan"])
unet   = UNet().to(device)
ddpm   = train_ddpm(GaussianDiffusion(unet, T=CFG["T"],
                    b0=CFG["beta_start"], b1=CFG["beta_end"]).to(device), EP["ddpm"])
clf    = train_mlp(MLPClassifier(), EP["mlp"])
print("semua model siap.")

---
# BAGIAN 3 — Pembuktian Batasan MLP Classifier (Tahap 1–5)

Forward classifier: `h1 = ReLU(x·W1+b1)`, `z_MLP = ReLU(h1·W2+b2)`, `logits = z_MLP·W3+b3`
dengan W1∈R^{784×256}, W2∈R^{256×64}, W3∈R^{64×10}. Proyeksi balik memakai transpose bobot
`nn.Linear.weight` (shape `out×in`) — sesuai rumus soal, bias diabaikan.

In [ ]:
# ===== Tahap 1: akurasi MLP Classifier =====
acc = test_acc(clf, test_loader)
print(f"MLPClassifier test accuracy = {acc*100:.2f}%  (syarat >= 88%)")
mlp_status = dict(test_accuracy=acc, threshold=0.88, meets_threshold=acc >= 0.88, quick=QUICK)
Path('results/mlp_status.json').write_text(json.dumps(mlp_status, indent=2))
if QUICK:
    print('SMOKE TEST: hasil ini bukan hasil final untuk laporan.')
if acc < 0.88:
    print('BELUM MEMENUHI syarat 88%. Analisis berikut tetap tersedia sebagai hasil diagnostik; jangan klaim syarat sudah tercapai.')

In [ ]:
# ===== Tahap 2: W^T projection dari one-hot kelas y_c =====
# z_hat = ReLU(y_c @ W3) ; h1_hat = ReLU(z_hat @ W2) ; x_hat = Sigmoid(h1_hat @ W1)
clf.eval()
with torch.no_grad():
    Y = torch.eye(10, device=device)                       # one-hot y_c, 10 kelas
    z_hat  = F.relu(Y @ clf.fc3.weight)                    # (10,64)
    h1_hat = F.relu(z_hat @ clf.fc2.weight)                # (10,256)
    x_hat  = torch.sigmoid(h1_hat @ clf.fc1.weight)        # (10,784)
wt_from_class = x_hat.view(10, 1, 28, 28).cpu()
# Normalisasi min-max per citra untuk visualisasi kontras (karena Sigmoid di sekitar 0 berada di ~[0.45, 0.55])
wt_min = wt_from_class.flatten(1).min(1)[0][:, None, None, None]
wt_max = wt_from_class.flatten(1).max(1)[0][:, None, None, None]
wt_from_class_vis = (wt_from_class - wt_min) / (wt_max - wt_min + 1e-8)
show_grid(wt_from_class_vis, nrow=10, size=8,
          title="Tahap 2: Proyeksi transpose bobot dari one-hot kelas")
plt.show()

In [ ]:
# ===== Tahap 3: latent transpose projection dari z_MLP citra test =====
x_test8, y_test8 = next(iter(test_loader))
x_test8 = x_test8[:8].to(device)
with torch.no_grad():
    z_mlp = clf.features(x_test8)                          # (8,64)
    h1_b  = F.relu(z_mlp @ clf.fc2.weight)                 # (8,256)
    x_b   = torch.sigmoid(h1_b @ clf.fc1.weight)           # (8,784)
zproj = x_b.view(8, 1, 28, 28).cpu()
fig, ax = plt.subplots(1, 2, figsize=(10, 2.6))
show_grid(x_test8.cpu(), 8, "Asli", ax[0])
show_grid(zproj, 8, "Tahap 3: Sigmoid(ReLU(z_MLP·W2ᵀ)·W1ᵀ)", ax[1])
plt.show()

In [ ]:
# ===== Tahap 4: Frozen Latent Decoding =====
# Bekukan classifier, latih decoder terpisah 64->256->784 (MSE) dari z_MLP beku.
for p in clf.parameters(): p.requires_grad_(False)
clf.eval()
fdec = FrozenDecoder().to(device)

def fdec_loss(model, x):
    with torch.no_grad():
        z = clf.features(x)
    return F.mse_loss(model(z), x), {}

fdec = train_ae_like(fdec, "fdec", EP["fdec"], fdec_loss)
print("frozen decoder selesai.")

In [ ]:
# ===== Tahap 5: prior sampling di ruang laten MLP =====
# mu_z, sigma_z per-dim dari z_MLP test set -> z_rand ~ N(mu, diag sigma^2) -> decode.
clf.eval(); fdec.eval()
zs = []
with torch.no_grad():
    for x, _ in test_loader:
        zs.append(clf.features(x.to(device)).cpu())
zs = torch.cat(zs)
mu_z, sd_z = zs.mean(0), zs.std(0)
with torch.no_grad():
    z_rand = mu_z.to(device) + sd_z.to(device) * torch.randn(16, 64, device=device)
    x_rand = fdec(z_rand).cpu()
show_grid(x_rand, nrow=4, size=5,
          title="Tahap 5: Sampling Gaussian diagonal dari statistik latent test")
plt.show()

---
# BAGIAN 4 — Evaluasi Kuantitatif (Tabel Metrik)

Metrik: Test MSE / PSNR / SSIM (model rekonstruktif), FID (model generatif, N sampel vs test set),
latency per batch-64 (CUDA-timed), dan training time/epoch dari log CSV.

In [ ]:
# ===== Fungsi metrik =====
@torch.no_grad()
def eval_recon(model, loader, kind="ae"):
    """Return dict mse/psnr/ssim; VAE memakai posterior mean untuk rekonstruksi deterministik."""
    model.eval()
    mses, psnrs, ssims = [], [], []
    for x, _ in loader:
        x = x.to(device)
        if kind == "ae":    r = model(x)
        elif kind == "vae": r = model.decode(model.encode(x)[0])
        else:             r = model(clf.features(x))         # fdec dari z_MLP
        e = ((r - x) ** 2).view(x.size(0), -1).mean(1)       # per-image MSE
        mses += e.tolist()
        psnrs += (10 * torch.log10(1.0 / e.clamp_min(1e-10))).tolist()
        rc, xc = r.cpu().numpy()[:, 0], x.cpu().numpy()[:, 0]
        ssims += [ssim_fn(a, b, data_range=1.0) for a, b in zip(rc, xc)]
    return dict(mse=np.mean(mses), psnr=np.mean(psnrs), ssim=np.mean(ssims))

@torch.inference_mode()
def measure_latency(fn, n_warm=3, n_iter=10):
    """fn() -> batch 64 gambar; CUDA-synced timing (ms)."""
    for _ in range(n_warm): fn()
    if device.type == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n_iter): fn()
    if device.type == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / n_iter * 1000

@torch.no_grad()
def gen_samples(tag, n, chunk=256):
    """n sampel [0,1] dari model generatif."""
    {'vae': vae, 'gan': ganG, 'dcgan': dcgG, 'ddpm': ddpm}[tag].eval()
    outs = []
    while sum(o.size(0) for o in outs) < n:
        m = min(chunk, n - sum(o.size(0) for o in outs))
        if tag == "vae":
            outs.append(vae.decode(torch.randn(m, CFG["latent"], device=device)).cpu())
        elif tag == "gan":
            outs.append(((ganG(torch.randn(m, CFG["gan_z"], device=device)) + 1) / 2).cpu())
        elif tag == "dcgan":
            outs.append(((dcgG(torch.randn(m, CFG["gan_z"], device=device)) + 1) / 2).cpu())
        elif tag == "ddpm":
            outs.append(ddpm.sample(m).cpu())
    return torch.cat(outs)[:n].clamp(0, 1)

In [ ]:
# ===== FID (InceptionV3 2048-d + scipy.linalg.sqrtm; tanpa dependensi eksternal) =====
from torchvision.models import inception_v3, Inception_V3_Weights

inception = inception_v3(weights=Inception_V3_Weights.DEFAULT, transform_input=True).to(device).eval()
inception.fc = nn.Identity()
for p in inception.parameters():
    p.requires_grad_(False)

@torch.inference_mode()
def extract_inception_features(imgs, net, batch_size=64):
    feats = []
    for b in imgs.split(batch_size):
        b = F.interpolate(b.to(device).clamp(0, 1), size=(299, 299), mode="bilinear", align_corners=False)
        b = b.repeat(1, 3, 1, 1)
        feats.append(net(b).cpu().numpy())
    return np.concatenate(feats, axis=0)

def calc_fid_from_feats(f1, f2):
    m1, s1 = f1.mean(axis=0), np.cov(f1, rowvar=False)
    m2, s2 = f2.mean(axis=0), np.cov(f2, rowvar=False)
    covmean = scipy.linalg.sqrtm(s1 @ s2)
    if np.iscomplexobj(covmean):
        covmean = covmean.real
    return float(np.sum((m1 - m2) ** 2) + np.trace(s1 + s2 - 2.0 * covmean))

print(f"hitung fitur InceptionV3 test set ({FID_N} sampel)...")
real_imgs = []
for x, _ in test_loader:
    real_imgs.append(x)
    if sum(t.size(0) for t in real_imgs) >= FID_N:
        break
real_feats = extract_inception_features(torch.cat(real_imgs)[:FID_N], inception)

fid_scores = {}
for tag in ["vae", "gan", "dcgan", "ddpm"]:
    print(f"generate {FID_N} sampel {tag}...")
    fake = gen_samples(tag, FID_N, chunk=256)
    fake_feats = extract_inception_features(fake, inception)
    fid_scores[tag] = calc_fid_from_feats(real_feats, fake_feats)
    print(f"  FID({tag}) = {fid_scores[tag]:.2f}")
if device.type == 'cuda': torch.cuda.empty_cache()

In [ ]:
# ===== Latency per batch 64 =====
for model in [ae, sparse, vae, ganG, dcgG, ddpm, clf, fdec]:
    model.eval()
lat = {}
# AE-family: latency = rekonstruksi forward; VAE/GAN/DCGAN/DDPM: latency = generasi 64 sampel
lat["ae"]    = measure_latency(lambda: ae(torch.zeros(64,1,28,28,device=device)))
lat["sparse"]= measure_latency(lambda: sparse(torch.zeros(64,1,28,28,device=device)))
lat["vae"]   = measure_latency(lambda: vae.decode(torch.randn(64,CFG["latent"],device=device)))
lat["gan"]   = measure_latency(lambda: ganG(torch.randn(64,CFG["gan_z"],device=device)))
lat["dcgan"] = measure_latency(lambda: dcgG(torch.randn(64,CFG["gan_z"],device=device)))
lat["ddpm"]  = measure_latency(lambda: ddpm.sample(64), n_warm=0, n_iter=2)
lat["mlp"]   = measure_latency(lambda: clf(torch.zeros(64,1,28,28,device=device)))
lat["fdec"]  = measure_latency(lambda: fdec(torch.randn(64,64,device=device)))
{print(k, f"{v:.1f} ms") for k, v in lat.items()}

In [ ]:
# ===== Susun tabel metrik (kolom sesuai template soal) =====
rec_ae  = eval_recon(ae, test_loader, "ae")
rec_sp  = eval_recon(sparse, test_loader, "ae")
rec_vae = eval_recon(vae, test_loader, "vae")
rec_fd  = eval_recon(fdec, test_loader, "fdec")

def epoch_sec(tag):
    df = pd.read_csv(f"results/logs/{tag}.csv"); return df["sec"].mean()
def logdf(tag): return pd.read_csv(f"results/logs/{tag}.csv")

# Tabel persis mengikuti template rubrik (tanpa kolom SSIM, sesuai PDF)
rows = [
 ["Vanilla Autoencoder", CFG["latent"], epoch_sec("ae"), lat["ae"],
  rec_ae["mse"], rec_ae["psnr"], "N/A (Latent Voids)", "Stabil / Konvergen", round(logdf("ae")["val_loss"].iloc[-1], 4)],
 ["Sparse Autoencoder", CFG["sparse_lat"], epoch_sec("sparse"), lat["sparse"],
  rec_sp["mse"], rec_sp["psnr"], "N/A (Latent Voids)", "Stabil / Konvergen", round(logdf("sparse")["val_loss"].iloc[-1], 4)],
 ["Variational Autoencoder (VAE)", CFG["latent"], epoch_sec("vae"), lat["vae"],
  rec_vae["mse"], rec_vae["psnr"], fid_scores["vae"], "Stabil / Konvergen", round(logdf("vae")["val_loss"].iloc[-1], 4)],
 ["GAN (Vanilla, MLP)", CFG["gan_z"], epoch_sec("gan"), lat["gan"],
  "N/A (Tanpa Encoder)", "N/A", fid_scores["gan"], "Sangat Rentan Osilasi / Mode Collapse", "N/A"],
 ["DCGAN", CFG["gan_z"], epoch_sec("dcgan"), lat["dcgan"],
  "N/A (Tanpa Encoder)", "N/A", fid_scores["dcgan"], "Cukup Stabil (Sensitif Arsitektur)", "N/A"],
 ["DDPM", "28x28", epoch_sec("ddpm"), lat["ddpm"],
  "N/A (Denoising Model)", "N/A", fid_scores["ddpm"], "Sangat Stabil", round(logdf("ddpm")["train_loss"].iloc[-1], 4)],
 ["MLP Classifier (referensi)", 64, epoch_sec("mlp"), lat["mlp"],
  "-", "-", "N/A", "Stabil", f"val_acc={logdf('mlp')['val_acc'].iloc[-1]:.4f}"],
 ["FrozenDecoder(z_MLP)", 64, epoch_sec("fdec"), lat["fdec"],
  rec_fd["mse"], rec_fd["psnr"], "N/A", "Stabil", round(logdf("fdec")["val_loss"].iloc[-1], 4)],
]
cols = ["Model", "Dim Laten", "Training Time / Epoch (s)", "Latency (ms / batch 64)",
        "Test MSE", "Test PSNR (dB)", "FID Score", "Kestabilan Training", "Final Val Loss/Acc"]
mt = pd.DataFrame(rows, columns=cols)
mt['Mode'] = 'SMOKE - bukan hasil final' if QUICK else 'FULL'
comparison = pd.DataFrame([rec_ae, rec_fd], index=['VanillaAE', 'FrozenDecoder'])
comparison.to_csv('results/mlp_reconstruction_comparison.csv')
print('Tahap 4: bandingkan metrik ini dengan validation loss; hasil buruk tidak boleh diasumsikan.')
print(comparison)
mt.to_csv("results/metrics.csv", index=False)
mt.to_json("results/metrics.json", orient="records", indent=2)
mt

---
# BAGIAN 5 — Lima Figur Wajib (disimpan ke `results/figures/`)

In [ ]:
# ===== Figur 1: Grid rekonstruksi GT / VanillaAE / SparseAE / VAE =====
ae.eval(); sparse.eval(); vae.eval()
with torch.no_grad():
    r_ae = ae(x_test8).cpu(); r_sp = sparse(x_test8).cpu()
    r_vae = vae.decode(vae.encode(x_test8)[0]).cpu()
fig, ax = plt.subplots(4, 1, figsize=(9, 6.2))
for a, imgs, t in zip(ax, [x_test8.cpu(), r_ae, r_sp, r_vae],
                      ["Ground Truth", "Vanilla AE", "Sparse AE", "VAE"]):
    show_grid(imgs, 8, t, a)
fig.suptitle("Figur 1 — Rekonstruksi side-by-side", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
fig.savefig("results/figures/fig1_reconstruction.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Figur 2: Random sampling 4x4 per model generatif =====
seed_everything(42)
s_vae = gen_samples("vae", 16); s_gan = gen_samples("gan", 16)
s_dcg = gen_samples("dcgan", 16); s_ddm = gen_samples("ddpm", 16)
fig, ax = plt.subplots(1, 4, figsize=(12, 3.4))
for a, imgs, t in zip(ax, [s_vae, s_gan, s_dcg, s_ddm],
                      ["A: VAE z~N(0,I)", "B: Vanilla GAN", "C: DCGAN", "D: DDPM"]):
    show_grid(imgs, 4, t, a, 3)
fig.suptitle("Figur 2 — Random generative sampling", y=1.02)
fig.savefig("results/figures/fig2_sampling.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Figur 3: Bukti empiris batasan MLP Classifier =====
with torch.no_grad():
    fd8 = fdec(clf.features(x_test8)).cpu()          # frozen decoder dari z_MLP
    ae8 = ae(x_test8).cpu()                          # pembanding z_AE
fig, ax = plt.subplots(2, 2, figsize=(11, 5.8))
show_grid(x_test8.cpu(), 8, "(1) Ground Truth (8 sampel test)", ax[0, 0])
show_grid(torch.cat([wt_from_class_vis[:8], zproj]), 8,
          "(2) W^T: atas = one-hot kelas, bawah = z_MLP", ax[0, 1])
show_grid(torch.cat([fd8, ae8]), 8,
          "(3) FrozenDecoder(z_MLP) atas vs VanillaAE bawah", ax[1, 0])
show_grid(x_rand[:8], 8, "(4) Prior sampling z_rand ~ N(mu_z, diag sigma_z^2)", ax[1, 1])
fig.suptitle("Figur 3 — Batasan bobot & latent MLP Classifier", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig("results/figures/fig3_mlp_proof.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Figur 4: Interpolasi latent alpha in {0,...,1} =====
alphas = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
# dua citra test dari kelas BERBEDA (kontras)
xA = xB = lA = None
for i in range(len(ds_test)):
    im, lb = ds_test[i]
    if xA is None:
        xA, lA = im, lb; continue
    if lb != lA:
        xB = im; break
xA, xB = xA[None].to(device), xB[None].to(device)
with torch.no_grad():
    zA_ae, zB_ae = ae.encode(xA), ae.encode(xB)
    zA_sp, zB_sp = sparse.encode(xA), sparse.encode(xB)
    zA_vae, zB_vae = vae.encode(xA)[0], vae.encode(xB)[0]
    zA_g, zB_g = torch.randn(1, CFG["gan_z"], device=device), torch.randn(1, CFG["gan_z"], device=device)

rows = []
for a in alphas:
    with torch.no_grad():
        interpolated = torch.cat([
        ae.decode((1 - a) * zA_ae + a * zB_ae),
        sparse.decode((1 - a) * zA_sp + a * zB_sp),
        vae.decode((1 - a) * zA_vae + a * zB_vae),
        ((ganG((1 - a) * zA_g + a * zB_g) + 1) / 2),
        ((dcgG((1 - a) * zA_g + a * zB_g) + 1) / 2)])
    rows.append(interpolated.cpu())
# susun per-model per-alpha -> grid 5 baris x 6 kolom
rows = torch.stack(rows)                      # (6 alpha, 5 model, 1,28,28)
grid_imgs = rows.permute(1, 0, 2, 3, 4).reshape(5 * 6, 1, 28, 28)
fig, ax = plt.subplots(figsize=(9, 8))
g = torchvision.utils.make_grid(grid_imgs, nrow=6, pad_value=0.5).permute(1, 2, 0).squeeze(-1)
ax.imshow(g, cmap="gray", vmin=0, vmax=1); ax.axis("off")
for i, name in enumerate(["VanillaAE", "SparseAE", "VAE", "GAN", "DCGAN"]):
    ax.annotate(name, xy=(-0.015, 1 - (i + 0.5) / 5), xycoords="axes fraction",
                rotation=90, va="center", ha="right", fontsize=9, annotation_clip=False)
for j, a in enumerate(alphas):
    ax.annotate(f"alpha={a}", xy=((j + 0.5) / 6, 1.01), xycoords="axes fraction",
                ha="center", fontsize=9, annotation_clip=False)
ax.set_title("Figur 4 — Linear latent interpolation", pad=24)
fig.savefig("results/figures/fig4_interpolation.png", dpi=220, bbox_inches="tight")
plt.show()

In [ ]:
# ===== Figur 5: Loss curves + DDPM denoising trajectory =====
def logdf(tag): return pd.read_csv(f"results/logs/{tag}.csv")

fig = plt.figure(figsize=(15, 7))
gs = fig.add_gridspec(2, 4, height_ratios=[1.4, 1])
ax1 = fig.add_subplot(gs[0, 0])
for tag, lb, col in [("ae", "VanillaAE", "tab:blue"), ("sparse", "SparseAE", "tab:orange")]:
    d = logdf(tag); ax1.plot(d.epoch, d.train_loss, marker=".", color=col, label=lb)
ax1.set_ylabel("MSE (AE / SparseAE)"); ax1.set_xlabel("epoch")
ax1b = ax1.twinx()
dv = logdf("vae"); ax1b.plot(dv.epoch, dv.train_loss, marker=".", color="tab:green", label="VAE (ELBO)")
ax1b.set_ylabel("ELBO (VAE)", color="tab:green")
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right", fontsize=8)
ax1.set_title("Reconstruction loss")

ax2 = fig.add_subplot(gs[0, 1]); d = logdf("gan")
ax2.plot(d.epoch, d.g_loss, marker=".", label="G"); ax2.plot(d.epoch, d.d_loss, marker=".", label="D")
ax2.set_title("Vanilla GAN"); ax2.legend(); ax2.set_xlabel("epoch")

ax3 = fig.add_subplot(gs[0, 2]); d = logdf("dcgan")
ax3.plot(d.epoch, d.g_loss, marker=".", label="G"); ax3.plot(d.epoch, d.d_loss, marker=".", label="D")
ax3.set_title("DCGAN"); ax3.legend(); ax3.set_xlabel("epoch")

ax4 = fig.add_subplot(gs[0, 3])
ax4.plot(logdf("ddpm").epoch, logdf("ddpm").train_loss, marker=".", color="tab:red")
ax4.set_title("DDPM noise-pred loss"); ax4.set_xlabel("epoch")

axt = fig.add_subplot(gs[1, :])
_, frames = ddpm.sample(1, traj=True)
traj = torchvision.utils.make_grid(torch.cat(frames), nrow=len(frames))
axt.imshow(traj.permute(1, 2, 0).squeeze(-1), cmap="gray", vmin=0, vmax=1)
axt.axis("off"); axt.set_title("DDPM reverse diffusion: x_T -> x_0 (kiri -> kanan)")

fig.suptitle("Figur 5 — Training loss & DDPM denoising trajectory", y=1.0)
fig.tight_layout()
fig.savefig("results/figures/fig5_losses_ddpm.png", dpi=220, bbox_inches="tight")
plt.show()

---
# Export hasil

Zip `checkpoints/`, `results/` (logs, metrics.csv/json, figures) lalu download otomatis di Colab.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
archive = RUN_DIR / 'genmodels_results.zip'
with ZipFile(archive, 'w', ZIP_DEFLATED) as output:
    for folder in ['checkpoints', 'results']:
        for path in sorted((RUN_DIR / folder).rglob('*')):
            if path.is_file():
                output.write(path, path.relative_to(RUN_DIR).as_posix())
print(f'{archive} siap: checkpoints + logs + metrics + figures + konfigurasi.')
if IN_COLAB:
    from google.colab import files
    files.download(str(archive))